<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-01-logica/03%20-%20Tautologias%20e%20Contradicoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook: Variação Lógica das Tags e Teste de Tautologias/Contradições
## Célula de Manufatura Flexível (FMS) - ECAA08

Este notebook gera valores lógicos e aleatórios para as variáveis de processo da planta de manufatura flexível e avalia as equações de intertravamento, inconsistências de sensores e provas de segurança.

In [1]:
import itertools
import random
import pandas as pd

# 1. Variáveis discretas mapeadas na planta de triagem flexível
variaveis = [
    'A', 'B', 'C_R', 'C_G', 'C_B', 'FC_p1', 'FC_p2', 'FC_p3',
    'S_vazio', 'S_p_saida', 'S_emerg', 'S_sobrecarga', 'e1',
    'ALM_cx1', 'ALM_cx2', 'ALM_cx3'
]

def gerar_estado_aleatorio():
    return {v: random.choice([True, False]) for v in variaveis}

# Geração exaustiva de todas as combinações booleanas (2^N)
todas_as_variacoes = [
    dict(zip(variaveis, valores))
    for valores in itertools.product([False, True], repeat=len(variaveis))
]

print(f'Total de variáveis analisadas: {len(variaveis)}')
print(f'Total de combinações no espaço de estados: {len(todas_as_variacoes)}')
print('\nExemplo de estado aleatório gerado:')
print(gerar_estado_aleatorio())

Total de variáveis analisadas: 16
Total de combinações no espaço de estados: 65536

Exemplo de estado aleatório gerado:
{'A': False, 'B': True, 'C_R': False, 'C_G': True, 'C_B': True, 'FC_p1': False, 'FC_p2': False, 'FC_p3': True, 'S_vazio': True, 'S_p_saida': True, 'S_emerg': False, 'S_sobrecarga': True, 'e1': False, 'ALM_cx1': True, 'ALM_cx2': False, 'ALM_cx3': True}


## 2. Modelagem das Funções e Equações Booleanas do SCADA

In [2]:
# Inconsistência física no sensor de geometria (Topo ativo sem a Base)
def inconsistencia_geometria(vars_):
    return (not vars_['A']) and vars_['B']

# Ambiguidade óptica (mais de uma cor ativada ao mesmo tempo)
def ambiguidade_cor(vars_):
    return (vars_['C_R'] and vars_['C_G']) or \
           (vars_['C_R'] and vars_['C_B']) or \
           (vars_['C_G'] and vars_['C_B'])

# Inconsistência no silo (vazio mas com peça na saída)
def inconsistencia_silo(vars_):
    return vars_['S_vazio'] and vars_['S_p_saida']

# Transbordo: todas as caixas cheias
def caixas_cheias(vars_):
    return vars_['ALM_cx1'] and vars_['ALM_cx2'] and vars_['ALM_cx3']

# Equação Geral de Alarme e Falha (HS-302 / a1)
def alarme_geral_a1(vars_):
    return vars_['e1'] or \
           (not vars_['S_emerg']) or \
           vars_['S_sobrecarga'] or \
           inconsistencia_geometria(vars_) or \
           ambiguidade_cor(vars_) or \
           inconsistencia_silo(vars_) or \
           caixas_cheias(vars_)

# Comandos de Triagem com Intertravamento de Segurança (bloqueados se a1 for True)
def cmd_pistao1(vars_):
    return vars_['A'] and vars_['B'] and vars_['C_R'] and (not vars_['FC_p1']) and (not alarme_geral_a1(vars_))

def cmd_pistao2(vars_):
    return vars_['A'] and vars_['B'] and vars_['C_G'] and (not vars_['FC_p2']) and (not alarme_geral_a1(vars_))

def cmd_pistao3(vars_):
    return vars_['A'] and (not vars_['B']) and vars_['C_B'] and (not vars_['FC_p3']) and (not alarme_geral_a1(vars_))

# Prova de Risco Proibido: Atuador ligado sob estado de alarme (deve ser Contradição)
def estado_risco_p1(vars_):
    return cmd_pistao1(vars_) and alarme_geral_a1(vars_)

# Prova Formal de Segurança: Negação do estado de risco (deve ser Tautologia)
def tautologia_seguranca_p1(vars_):
    return not (cmd_pistao1(vars_) and alarme_geral_a1(vars_))

# Tautologia e Contradição clássicas
def tautologia_clasica(vars_):
    return vars_['A'] or (not vars_['A'])

def contradicao_clasica(vars_):
    return vars_['A'] and (not vars_['A'])

# Dicionário de testes lógicos
expressoes = {
    '1. Inconsistência Geométrica (~A and B)': inconsistencia_geometria,
    '2. Ambiguidade de Sensores de Cor': ambiguidade_cor,
    '3. Inconsistência de Alimentação do Silo': inconsistencia_silo,
    '4. Disparo do Alarme Geral (a1)': alarme_geral_a1,
    '5. Comando Pistão 1 (Intertravado)': cmd_pistao1,
    '6. Comando Pistão 2 (Intertravado)': cmd_pistao2,
    '7. Comando Pistão 3 (Intertravado)': cmd_pistao3,
    '8. Estado Proibido (CMD_p1 AND a1)': estado_risco_p1,
    '9. Prova de Tautologia [~(CMD_p1 AND a1)]': tautologia_seguranca_p1,
    '10. Tautologia clássica (A or ~A)': tautologia_clasica,
    '11. Contradição clássica (A and ~A)': contradicao_clasica
}

def classificar_expressao(fn):
    valores = [fn(v) for v in todas_as_variacoes]
    if all(valores):
        return 'Tautologia'
    elif not any(valores):
        return 'Contradição'
    return 'Contingente'

resultado = []
for nome, fn in expressoes.items():
    valores = [fn(v) for v in todas_as_variacoes]
    resultado.append({
        'Expressão / Regra Lógica': nome,
        'Verdadeiras': sum(valores),
        'Falsas': len(valores) - sum(valores),
        'Classificação': classificar_expressao(fn)
    })

df_resultado = pd.DataFrame(resultado)
print(df_resultado.to_string(index=False))

                 Expressão / Regra Lógica  Verdadeiras  Falsas Classificação
  1. Inconsistência Geométrica (~A and B)        16384   49152   Contingente
        2. Ambiguidade de Sensores de Cor        32768   32768   Contingente
 3. Inconsistência de Alimentação do Silo        16384   49152   Contingente
          4. Disparo do Alarme Geral (a1)        63520    2016   Contingente
       5. Comando Pistão 1 (Intertravado)           84   65452   Contingente
       6. Comando Pistão 2 (Intertravado)           84   65452   Contingente
       7. Comando Pistão 3 (Intertravado)           84   65452   Contingente
       8. Estado Proibido (CMD_p1 AND a1)            0   65536   Contradição
9. Prova de Tautologia [~(CMD_p1 AND a1)]        65536       0    Tautologia
        10. Tautologia clássica (A or ~A)        65536       0    Tautologia
      11. Contradição clássica (A and ~A)            0   65536   Contradição


## 3. Simulação de Estados Aleatórios do Processo

In [3]:
for i in range(5):
    estado = gerar_estado_aleatorio()
    print(f'\n--- Simulação de Estado #{i+1} ---')
    print(f'Estado das Tags: {estado}')
    print('Alarme Geral (a1):', alarme_geral_a1(estado))
    print('CMD Pistão 1:', cmd_pistao1(estado))
    print('CMD Pistão 2:', cmd_pistao2(estado))
    print('CMD Pistão 3:', cmd_pistao3(estado))
    print('Validação de Segurança (Tautologia):', tautologia_seguranca_p1(estado))


--- Simulação de Estado #1 ---
Estado das Tags: {'A': True, 'B': True, 'C_R': False, 'C_G': False, 'C_B': True, 'FC_p1': True, 'FC_p2': True, 'FC_p3': False, 'S_vazio': False, 'S_p_saida': False, 'S_emerg': False, 'S_sobrecarga': False, 'e1': True, 'ALM_cx1': False, 'ALM_cx2': False, 'ALM_cx3': True}
Alarme Geral (a1): True
CMD Pistão 1: False
CMD Pistão 2: False
CMD Pistão 3: False
Validação de Segurança (Tautologia): True

--- Simulação de Estado #2 ---
Estado das Tags: {'A': False, 'B': True, 'C_R': False, 'C_G': True, 'C_B': True, 'FC_p1': True, 'FC_p2': True, 'FC_p3': False, 'S_vazio': False, 'S_p_saida': False, 'S_emerg': False, 'S_sobrecarga': True, 'e1': True, 'ALM_cx1': False, 'ALM_cx2': True, 'ALM_cx3': False}
Alarme Geral (a1): True
CMD Pistão 1: False
CMD Pistão 2: False
CMD Pistão 3: False
Validação de Segurança (Tautologia): True

--- Simulação de Estado #3 ---
Estado das Tags: {'A': False, 'B': True, 'C_R': True, 'C_G': True, 'C_B': True, 'FC_p1': True, 'FC_p2': False, 